# Midas Design Guide — Steel Composite Girder Design

**Companion notebook** for the corresponding chapter of the MIDAS training
manual *Design Guide for midas Civil — AASHTO LRFD*. The guide itself is
proprietary and is **not reproduced here** — this notebook contains only
original code and AASHTO LRFD / MBE article citations.

**How this notebook is used.** Work through the design process with this
notebook alongside: each step carries its AASHTO background in original
words, a live Python environment for exploratory checks and quick
validation, a direct interface to Midas Civil through its API, and
customization to ODOT's design process (PSID / PSBD standard products,
ODOT materials and vehicles).
Everything the guide has you do by hand in the Midas UI — coordinate entry,
side math — is demonstrated here as runnable code a designer can follow,
rerun, and modify. Where a number must come from the
Midas model itself, it is either pulled live over the Civil NX JSON API
(the verified result tables run in place) or recorded from the companion
six-beam model with its provenance noted; the remaining `TODO(midas-api)`
markers await the PSC design module and construction-stage analysis. Hand-entered values (from a standard
drawing or a hand calc) are compared via the `check()` harness below.

**Scope of this chapter:**

1. Bridge description, girder plate sizes, and materials from the guide example
2. Section proportion limits (LRFD 6.10.2)
3. Composite section properties (n, 3n, cracked) and effective flange width
4. Live load distribution factors and force effects (API)
5. Constructibility — LRFD 6.10.3
6. Service II — permanent deformations, LRFD 6.10.4
7. Fatigue — LRFD 6.10.5 / 6.6.1
8. Strength I flexure — positive (6.10.7) and negative (6.10.8 / Appendix A6)
9. Shear and transverse stiffeners — LRFD 6.10.9 / 6.10.11
10. Shear connectors — LRFD 6.10.10
11. Bearing stiffeners and cross-check vs Midas composite design tables (API)


In [ ]:
import math
import pandas as pd

# civilpy is installed editable into this env (pip install -e .)
from civilpy.structural.midas import MidasCivil, parse_result_table, envelope
from civilpy.structural.aashto.lrfd import (
    concrete, prestressed, steel, composite, distribution, lrfr, creep_shrinkage,
)

# --- Midas Civil NX connection -------------------------------------------------
# The API is not running on this machine right now. Everything below that needs
# the live model is guarded by MIDAS_ONLINE and marked TODO(midas-api).
try:
    midas = MidasCivil()
    MIDAS_ONLINE = midas.ping()
except Exception:
    midas, MIDAS_ONLINE = None, False
print("Midas Civil NX online:", MIDAS_ONLINE)

In [ ]:
# --- Validation harness --------------------------------------------------------
# Every comparison in this notebook goes through check() so the end-of-notebook
# summary shows guide value vs civilpy value side by side.
RESULTS = []

def check(label, guide_value, civilpy_value, tol=0.01, unit=""):
    """Compare a guide-reported value against the civilpy-computed one.

    tol is relative (1% default) — the guide rounds intermediate values, so
    small drift is expected; flag anything beyond tol for investigation.
    """
    if guide_value is None or civilpy_value is None:
        status = "PENDING"
        diff = None
    else:
        diff = abs(civilpy_value - guide_value) / (abs(guide_value) or 1.0)
        status = "OK" if diff <= tol else "MISMATCH"
    RESULTS.append({"check": label, "guide": guide_value, "civilpy": civilpy_value,
                    "rel diff": diff, "unit": unit, "status": status})
    print(f"[{status}] {label}: guide={guide_value} civilpy={civilpy_value} {unit}")
    return status == "OK"

def summary():
    df = pd.DataFrame(RESULTS)
    if len(df):
        n_ok = (df.status == "OK").sum()
        print(f"{n_ok}/{len(df)} checks OK, "
              f"{(df.status == 'MISMATCH').sum()} mismatches, "
              f"{(df.status == 'PENDING').sum()} pending")
    return df

## 1. Design inputs from the guide example

In [ ]:
# TODO(guide): fill from the chapter's worked example.
GUIDE = dict(
    spans_ft=None,           # e.g. [90, 120, 90] for the continuous example
    girder_spacing_ft=None,
    n_girders=None,
    deck_thickness_in=None,
    haunch_in=None,
    fy_ksi=50.0,
    fc_deck_ksi=None,
    # plate schedule: list of (region, top flange bxt, web DxT, bottom flange bxt)
    plates=None,
)
GUIDE

## 2. Section proportion limits — LRFD 6.10.2

Web slenderness without longitudinal stiffeners, flange proportion limits
(b/2t, flange-to-web ratios, Iyc/Iyt).

In [ ]:
# TODO(guide): steel.proportion_limits(...) per plate region; check() each
# ratio the guide reports.
pass

## 3. Composite section properties

Short-term (n), long-term (3n), and cracked-section properties per region;
effective flange width per 4.6.2.6.

In [ ]:
# TODO(guide):
# n = composite.modular_ratio(fc=...)
# girder = composite.CompositeGirder(...)  # per plate region
# check() section moduli against the guide's tables.
pass

## 4. Distribution factors and force effects

DFs per 4.6.2.2 (steel I-girder row), then the unfactored envelopes from the
Midas model — DC1/DC2/DW split matters for the 3n vs n stress buildup.

In [ ]:
# TODO(guide): distribution.moment_df_interior(...) etc., as in the PSC
# chapter, then:

In [ ]:
if MIDAS_ONLINE:
    # Verified vs live Civil NX 2026-07-27: /post/TABLE selects by
    # TABLE_TYPE ("BEAMFORCE"); TABLE_NAME is just a label.
    try:
        resp = midas.result_table("BeamForce",
                                  table_type="BEAMFORCE")
        display(pd.DataFrame(parse_result_table(resp)).head())
    except Exception as err:
        # a fresh/unanalyzed session (or a DB edit, or a pre-mode view
        # switch) clears results — analyze and rerun this cell
        print("no results in the session:", str(err)[-80:])
else:
    print("Midas offline — skipping BeamForce (BEAMFORCE)")

## 5. Constructibility — LRFD 6.10.3

Deck-casting sequence stresses on the noncomposite section: flange nominal
yielding, flange local buckling, LTB with the casting unbraced lengths, and
web bend-buckling.

In [ ]:
# TODO(guide):
# steel.constructibility_compression_flange(...)
# steel.web_bend_buckling(...)
# steel.lateral_torsional_buckling_resistance(...)
# TODO(midas-api): construction-stage results table for the casting sequence.
pass

## 6. Service II — LRFD 6.10.4

Flange stress limits 0.95·Rh·Fyf (composite) / 0.80·Rh·Fyf, with the
1.3·LL+IM Service II combination.

In [ ]:
# TODO(guide): build f_f from the staged section moduli (DC1 on steel,
# DC2+DW on 3n, LL on n); steel.hybrid_factor(...) if hybrid.
pass

## 7. Fatigue — LRFD 6.10.5 / 6.6.1

Governing details (web-to-flange weld, stiffener welds, shear studs), fatigue
category resistance, and the Fatigue I/II stress ranges from the fatigue
truck.

In [ ]:
# TODO(guide): steel.fatigue_resistance(category=..., n_cycles=...)
# TODO(midas-api): fatigue truck moving-load stress ranges from the model.
pass

## 8. Strength I flexure

Positive flexure: compact composite section check and Dp/Dt ductility
(6.10.7). Negative flexure: FLB / LTB per 6.10.8, or Appendix A6 if the guide
uses it for the compact-web continuous section.

In [ ]:
# TODO(guide):
# steel.compact_composite_positive_flexure(...)
# steel.flange_local_buckling_resistance(...)
# steel.lateral_torsional_buckling_resistance(...)
# steel.tension_flange_resistance(...)
# civilpy.structural.aashto.lrfd.appendix_a6 for the A6 path if used.
pass

## 9. Shear — LRFD 6.10.9

End/interior panel shear with tension-field action, transverse stiffener
spacing, stiffener proportioning (6.10.11.1).

In [ ]:
# TODO(guide):
# steel.web_shear_resistance(...)
# steel.transverse_stiffener_width(...), steel.transverse_stiffener_inertia(...)
pass

## 10. Shear connectors — LRFD 6.10.10

Fatigue pitch governing the layout, strength check of total connectors between
points of max moment and zero moment.

In [ ]:
# TODO(guide):
# steel.shear_connector_strength(...)
# steel.shear_connector_fatigue_pitch(...)
pass

## 11. Bearing stiffeners and Midas cross-check

In [ ]:
# TODO(guide): steel.bearing_stiffener_resistance(...), bearing_stiffener_width(...)

In [ ]:
# TODO(midas-api): "Composite Girder Design Result" is a PSC *design-module* table.
# Verified 2026-07-27: the design result tables (fps, c, Mcr, Av,req,
# FDL/AFDL columns) are NOT exposed through the known /post/TABLE surface —
# they need the PSC Design run configured in the Civil NX UI (design code,
# PSC design parameters, Section Manager rebar) and/or the official JSON
# manual's design TABLE_TYPE names. Analysis-side tables ARE verified:
# BEAMFORCE, BEAMSTRESSPSC (the ten-check-point stress table with
# Sig-Is(shear), Sig-Is(shear+torsion), Sig-Ps(Max/Min) columns), REACTIONG.
print("PENDING: Composite Girder Design Result — needs the PSC Design module (see comment)")

## Validation summary

Every `check()` recorded above, in one table. `PENDING` rows are waiting on
either guide values (hand entry) or the Midas API coming back online.

In [ ]:
summary()